# Capitalisations à prendre
## Méga-capitalisations (groupe contrôle, 12)
- AAPL
- MSFT


## Intermédiaires

In [1]:
import yfinance as yf
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime
from time import sleep

In [2]:
# values

name='prices'
db_path = '../data/prices.db'
parquet_path = '../data/prices.parquet'
start="2025-01-01"
end="2025-12-31"

In [ ]:
# CORRECTION


def init_db(db_path):
    con = duckdb.connect(db_path)

    # create table
    con.execute("""
                CREATE TABLE IF NOT EXISTS prices ( 
                date DATE NOT NULL, 
                ticker VARCHAR NOT NULL, 
                adj_close DOUBLE NOT NULL,
                close DOUBLE NOT NULL, 
                high DOUBLE NOT NULL, 
                low DOUBLE NOT NULL,  
                open DOUBLE NOT NULL, 
                volume BIGINT NOT NULL,
                PRIMARY KEY (date, ticker)
                );
                """
    )

    # Save table

    con.close()

def fetch_prices(tickers, start, end):
    columns = ['date', 'ticker', 'adj_close', 'close', 'high', 'low', 'open','volume']
    df = pd.DataFrame(columns = columns)
    df.dtype = ['str', 'datetime64[ns]', 'float', 'float', 'float', 'float', 'float', 'int']

    for ticker in tickers:
        rows = yf.download(ticker,start=start, end=end, auto_adjust=False).dropna(axis=1, how='all').stack().reset_index()
        rows.columns = columns
        df = pd.concat([df, rows])
        sleep(0.3)
    return df

def store_prices(df, db_path):
    con = duckdb.connect(db_path)
    con.register('my_df', df)
    con.execute('INSERT OR REPLACE INTO prices SELECT * FROM my_df')
    con.close()
    
def query_prices(ticker, db_path) -> pd.DataFrame:
    with duckdb.connect(db_path) as con:
        query = 'SELECT * FROM prices WHERE ticker = ?'
        output = con.execute(query, [ticker]).fetchdf()
        con.close()

    return output

In [4]:
# testing

tickers = ['AAPL', 'MSFT', 'SOFI', 'LCID', 'GOOG', 'TTE']

init_db(db_path)
df = fetch_prices(tickers, start, end)
store_prices(df, db_path)
query_prices('LCID', db_path)


/tmp/ipykernel_117181/4152762850.py:30: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.dtype = ['str', 'datetime64[ns]', 'float', 'float', 'float', 'float', 'float', 'int']
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


            date ticker   adj_close       close        high         low  \
0     2025-01-02   AAPL  242.301926  243.850006  249.100006  241.820007   
1     2025-01-03   AAPL  241.815033  243.360001  244.179993  241.889999   
2     2025-01-06   AAPL  243.444595  245.000000  247.330002  243.199997   
3     2025-01-07   AAPL  240.672318  242.210007  245.550003  241.350006   
4     2025-01-08   AAPL  241.159210  242.699997  243.710007  240.050003   
...          ...    ...         ...         ...         ...         ...   
1489  2025-12-23    TTE   64.321968   66.010002   66.160004   65.650002   
1490  2025-12-24    TTE   63.863983   65.540001   66.190002   65.510002   
1491  2025-12-26    TTE   63.854237   65.529999   65.930000   65.190002   
1492  2025-12-29    TTE   64.214775   65.900002   66.150002   65.739998   
1493  2025-12-30    TTE   64.234261   65.919998   66.629997   65.919998   

            open    volume  
0     248.929993  55740700  
1     243.360001  40244100  
2     244.30

,date,ticker,adj_close,close,high,low,open,volume
0,2025-01-13,LCID,30.100000,30.100000,30.299999,28.500000,30.000000,7691080
1,2025-01-22,LCID,26.700001,26.700001,28.400000,26.700001,28.200001,8275450
2,2025-02-05,LCID,29.100000,29.100000,30.400000,29.000000,29.400000,6939050
3,2025-02-24,LCID,27.799999,27.799999,30.000000,27.100000,29.850000,11496170
4,2025-02-26,LCID,22.549999,22.549999,24.799999,22.500000,24.450001,16044440
...,...,...,...,...,...,...,...,...
244,2025-12-04,LCID,14.150000,14.150000,14.200000,13.510000,13.790000,7706400
245,2025-12-05,LCID,13.420000,13.420000,14.260000,13.400000,14.115000,9045700
246,2025-12-11,LCID,12.830000,12.830000,13.010000,12.660000,12.855000,3786100
247,2025-12-15,LCID,11.810000,11.810000,12.800000,11.800000,12.780000,7621400


In [5]:
test = yf.download('AAPL', start='2024-01-01', end='2024-02-01', auto_adjust=False)
print(test.columns)
print(test.stack().reset_index().head())

[*********************100%***********************]  1 of 1 completed

MultiIndex([('Adj Close', 'AAPL'),
            (    'Close', 'AAPL'),
            (     'High', 'AAPL'),
            (      'Low', 'AAPL'),
            (     'Open', 'AAPL'),
            (   'Volume', 'AAPL')],
           names=['Price', 'Ticker'])
Price    level_0 Ticker   Adj Close       Close        High         Low  \
0     2024-01-02   AAPL  183.562180  185.639999  188.440002  183.889999   
1     2024-01-03   AAPL  182.187744  184.250000  185.880005  183.429993   
2     2024-01-04   AAPL  179.873947  181.910004  183.089996  180.880005   
3     2024-01-05   AAPL  179.152084  181.179993  182.759995  180.169998   
4     2024-01-08   AAPL  183.483063  185.559998  185.600006  181.500000   

Price        Open    Volume  
0      187.149994  82488700  
1      184.220001  58414500  
2      182.149994  71983600  
3      181.990005  62379700  
4      182.089996  59144500  


Loop over tickers, pull each from yfinance, normalize into your chosen long format with consistent column names. Handle the cases where a ticker returns nothing or errors out (one bad ticker shouldn't kill the whole run). Be polite with a small sleep between requests. Heads-up on a real gotcha: recent yfinance versions return a MultiIndex column frame even for a single ticker — you'll need to flatten it, and discovering that yourself by inspecting df.columns is a good exercise.

## The spec — implement these four functions:

init_db(db_path) — connect to DuckDB, create a prices table if it doesn't exist with columns for ticker, date, the OHLC prices, adjusted close, and volume. Decide the primary key. Think about types (what's the right type for volume vs. price?).

fetch_prices(tickers, start, end) -> DataFrame — loop over tickers, pull each from yfinance, normalize into your chosen long format with consistent column names. Handle the cases where a ticker returns nothing or errors out (one bad ticker shouldn't kill the whole run). Be polite with a small sleep between requests. Heads-up on a real gotcha: recent yfinance versions return a MultiIndex column frame even for a single ticker — you'll need to flatten it, and discovering that yourself by inspecting df.columns is a good exercise.

store_prices(df, db_path) — write the dataframe into DuckDB idempotently. Look up how DuckDB lets you query a registered pandas dataframe directly; it's elegant and you'll use the pattern repeatedly.

query_prices(ticker, db_path) -> DataFrame — read one ticker's history back, date-ordered. Use a parameterized query, not string formatting — I want you to build the safe-SQL habit from day one even though this is local.
How to validate your work: write a small script that fetches maybe five tickers (mix a mega-cap like MSFT with something small and volatile like SOFI), stores them, reads one back, and prints the date range and row count. If you can round-trip real data, the layer works.

In [1]:
# init_db

# create a persistent DB

# create prices table : ticker, date, OHLC, adjusted close, volume. Primary key should be (ticket, date) ? or add an ID ?
# volume : int ; price : float
